# Week 2 - Preprocessing - Part 2
Keshia-Lee Martin<br>
OMDS-ModB2<br>
Media and Entertainment<br>

Datasets: <br>
[Movielens](https://grouplens.org/datasets/movielens/latest/), [Dataset Details](https://files.grouplens.org/datasets/movielens/ml-latest-small-README.html)<br>
[4 Services Streaming Movies and Tv](https://www.kaggle.com/datasets/sc0v1n0/4-services-streaming-movies-and-tv/data)

# Weekly graph question

The Storytelling With Data book mentions planning on a "Who, What, and How" for your data story.  Write down a possible Who, What, and How for your data, using the ideas in the book.

# Homework 2

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

## Movielens Dataset

The MovieLens dataset provides large-scale user-generated ratings, tags, and viewing preferences that capture audience sentiment, movie
popularity, and behavioral patterns associated with consumer engagement.

In [2]:
# creating DataFrames/variables for each csv file to understand the data in each file before combining
df_movie_links = pd.read_csv("movielens-links.csv")

df_movie_links.info()
print()
df_movie_links.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9742 non-null   int64  
 1   imdbId   9742 non-null   int64  
 2   tmdbId   9734 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 228.5 KB



,movieId,imdbId,tmdbId
count,9742.000000,9.742000e+03,9734.000000
mean,42200.353623,6.771839e+05,55162.123793
std,52160.494854,1.107228e+06,93653.481487
min,1.000000,4.170000e+02,2.000000
25%,3248.250000,9.518075e+04,9665.500000
50%,7300.000000,1.672605e+05,16529.000000
75%,76232.000000,8.055685e+05,44205.750000
max,193609.000000,8.391976e+06,525662.000000


In [3]:
df_movie_names = pd.read_csv("movielens-movies.csv")

df_movie_names.info()
print()
df_movie_names.describe()

# checking for movie duplicates - they have different movieIds, and different genre tags
df_movie_names['title'].is_unique
df_movie_names[df_movie_names['title'].duplicated(keep=False)]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB



,movieId,title,genres
650,838,Emma (1996),Comedy|Drama|Romance
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
4169,6003,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller
5601,26958,Emma (1996),Romance
5854,32600,Eros (2004),Drama
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller
6932,64997,War of the Worlds (2005),Action|Sci-Fi
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
9135,147002,Eros (2004),Drama|Romance
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller


In [4]:
# movie ratings by users
df_movie_ratings = pd.read_csv("movielens-ratings.csv")

df_movie_ratings.info()
print()
df_movie_ratings.describe() 
# based on data, need to check if the movieids are unique in the dataset.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB



,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


In [5]:
# users tags of the movies
df_movie_tags = pd.read_csv("movielens-tags.csv")

df_movie_tags.info()
print()
df_movie_tags.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   userId     3683 non-null   int64 
 1   movieId    3683 non-null   int64 
 2   tag        3683 non-null   object
 3   timestamp  3683 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 115.2+ KB



,userId,movieId,timestamp
count,3683.000000,3683.000000,3.683000e+03
mean,431.149335,27252.013576,1.320032e+09
std,158.472553,43490.558803,1.721025e+08
min,2.000000,1.000000,1.137179e+09
25%,424.000000,1262.500000,1.137521e+09
50%,474.000000,4454.000000,1.269833e+09
75%,477.000000,39263.000000,1.498457e+09
max,610.000000,193565.000000,1.537099e+09


In [6]:
# combining the movie links csv and the movie titles csvs first
links_name_merged = df_movie_links.merge(df_movie_names, on='movieId', how='outer')
links_name_merged.head()

,movieId,imdbId,tmdbId,title,genres
0,1,114709,862.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,113497,8844.0,Jumanji (1995),Adventure|Children|Fantasy
2,3,113228,15602.0,Grumpier Old Men (1995),Comedy|Romance
3,4,114885,31357.0,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,113041,11862.0,Father of the Bride Part II (1995),Comedy


In [7]:
# combined the ratings csv with the the movie tags csv
ratings_tags_merged = df_movie_ratings.merge(df_movie_tags, on='movieId', how='outer')
ratings_tags_merged.sample(5)

,userId_x,movieId,rating,timestamp_x,userId_y,tag,timestamp_y
161421,489.0,2594,4.0,1.333659e+09,NaN,NaN,NaN
37185,103.0,296,5.0,1.431954e+09,599.0,action,1.498456e+09
208652,610.0,4902,4.0,1.493846e+09,474.0,ghosts,1.137368e+09
62963,368.0,296,5.0,9.758275e+08,599.0,guns,1.498456e+09
42574,153.0,296,3.0,1.525549e+09,599.0,random,1.498457e+09


In [8]:
# merged previous combined csvs together
df_merge_movies = links_name_merged.merge(ratings_tags_merged, on='movieId', how='outer')
df_merge_movies.sample(5)

,movieId,imdbId,tmdbId,title,genres,userId_x,rating,timestamp_x,userId_y,tag,timestamp_y
256658,68157,361748,16869.0,Inglourious Basterds (2009),Action|Drama|War,209.0,5.0,1.524522e+09,424.0,satire,1.457845e+09
155829,2324,118799,637.0,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama|Romance|War,365.0,2.0,1.488333e+09,474.0,Holocaust,1.137181e+09
276758,106489,1170358,57158.0,"Hobbit: The Desolation of Smaug, The (2013)",Adventure|Fantasy|IMAX,10.0,3.5,1.455357e+09,106.0,adventure,1.467567e+09
56847,296,110912,680.0,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,304.0,5.0,8.911732e+08,599.0,nonlinear storyline,1.498457e+09
36350,296,110912,680.0,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,94.0,4.0,8.434066e+08,599.0,good music,1.498456e+09


Based on the html details about the data, the timestamps represent seconds since midnight Coordinated Universal Time (UTC) of January 1, 1970. There is no needs for this for our data. The timestamp columns need to be dropped.

In [9]:
df_merge_movies = df_merge_movies.drop('timestamp_x', axis=1)
df_merge_movies = df_merge_movies.drop('timestamp_y', axis=1)


Renaming columns userid_x and userid_y to know that which id is for the rating and which user is for the tagging of the movie.

In [10]:
df_merge_movies = df_merge_movies.rename(columns={'userId_x':'userId_rating'})
df_merge_movies = df_merge_movies.rename(columns={'userId_y':'userId_tag'})

Now, verifying that the columns were dropped and the columns were renamed.

In [11]:
df_merge_movies.sample(5)

,movieId,imdbId,tmdbId,title,genres,userId_rating,rating,userId_tag,tag
103453,597,100405,114.0,Pretty Woman (1990),Comedy|Romance,273.0,4.0,474.0,prostitution
164674,2717,97428,2978.0,Ghostbusters II (1989),Comedy|Fantasy|Sci-Fi,387.0,3.0,474.0,Ghosts
206052,4878,246578,141.0,Donnie Darko (2001),Drama|Mystery|Sci-Fi|Thriller,318.0,3.0,567.0,atmospheric
174870,2959,137523,550.0,Fight Club (1999),Action|Crime|Drama|Thriller,331.0,4.0,599.0,crime
50393,296,110912,680.0,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,231.0,4.0,424.0,Quentin Tarantino


The following steps, there will be checking for duplicate rows, missing or null values.

In [12]:
df_merge_movies.info()
print()
df_merge_movies.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 285783 entries, 0 to 285782
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   movieId        285783 non-null  int64  
 1   imdbId         285783 non-null  int64  
 2   tmdbId         285770 non-null  float64
 3   title          285783 non-null  object 
 4   genres         285783 non-null  object 
 5   userId_rating  285762 non-null  float64
 6   rating         285762 non-null  float64
 7   userId_tag     233234 non-null  float64
 8   tag            233234 non-null  object 
dtypes: float64(4), int64(2), object(3)
memory usage: 19.6+ MB



,movieId,imdbId,tmdbId,userId_rating,rating,userId_tag
count,285783.000000,2.857830e+05,285770.000000,285762.000000,285762.000000,233234.000000
mean,14927.663741,2.956050e+05,12797.315320,313.894279,3.841270,470.681354
std,31402.673519,5.150156e+05,43479.255523,179.451387,1.020798,153.324249
min,1.000000,4.170000e+02,2.000000,1.000000,0.500000,2.000000
25%,296.000000,1.098300e+05,489.000000,160.000000,3.000000,424.000000
50%,1721.000000,1.125730e+05,680.000000,314.000000,4.000000,477.000000
75%,5673.000000,2.415270e+05,8963.000000,465.000000,4.500000,599.000000
max,193609.000000,8.391976e+06,525662.000000,610.000000,5.000000,610.000000


In [13]:
# checking for duplicate rows
duplicated_rows = df_merge_movies[df_merge_movies.duplicated()]
duplicated_rows

,movieId,imdbId,tmdbId,title,genres,userId_rating,rating,userId_tag,tag


In [14]:
# checking for missing or null values
df_merge_movies_null = df_merge_movies[df_merge_movies.isnull().any(axis=1)]
df_merge_movies_null.sample(5)

,movieId,imdbId,tmdbId,title,genres,userId_rating,rating,userId_tag,tag
141131,1639,118842,2255.0,Chasing Amy (1997),Comedy|Drama|Romance,480.0,2.5,NaN,NaN
136957,1370,99423,1573.0,Die Hard 2 (1990),Action|Adventure|Thriller,561.0,3.0,NaN,NaN
210929,5025,273923,11857.0,Orange County (2002),Comedy,219.0,1.0,NaN,NaN
92595,442,106697,9739.0,Demolition Man (1993),Action|Adventure|Sci-Fi,134.0,4.0,NaN,NaN
243518,45668,410297,2044.0,"Lake House, The (2006)",Drama|Fantasy|Romance,517.0,2.0,NaN,NaN


Based on the findings above, some movies did not have a user that gave the movie a tag. I think these empty values can be ignored.

In [15]:
# mean, median, mode of rating, the only column it would make sense to do this for
ratings_mean = df_merge_movies['rating'].mean()
ratings_median = df_merge_movies['rating'].median()
ratings_mode = df_merge_movies['rating'].mode()[0]

print(f"Movie ratings mean: {ratings_mean}")
print(f"Movie ratings median: {ratings_median}")
print(f"Movie ratings mode: {ratings_mode}")

Movie ratings mean: 3.8412700079086792
Movie ratings median: 4.0
Movie ratings mode: 4.0


In [16]:
# categorical variables with one hot encoding
# look at value as a string, separate each string by the separator
one_hot = df_merge_movies['genres'].str.get_dummies(sep='|')
one_hot

,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
4,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285778,0,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
285779,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
285780,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
285781,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [17]:
# making a duplicate of the merged movies dataframe so it doesn't accidently change the original
df_merge_movies_final = df_merge_movies.copy()
df_merge_movies_final = df_merge_movies_final.join(one_hot)
df_merge_movies_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 285783 entries, 0 to 285782
Data columns (total 29 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   movieId             285783 non-null  int64  
 1   imdbId              285783 non-null  int64  
 2   tmdbId              285770 non-null  float64
 3   title               285783 non-null  object 
 4   genres              285783 non-null  object 
 5   userId_rating       285762 non-null  float64
 6   rating              285762 non-null  float64
 7   userId_tag          233234 non-null  float64
 8   tag                 233234 non-null  object 
 9   (no genres listed)  285783 non-null  int64  
 10  Action              285783 non-null  int64  
 11  Adventure           285783 non-null  int64  
 12  Animation           285783 non-null  int64  
 13  Children            285783 non-null  int64  
 14  Comedy              285783 non-null  int64  
 15  Crime               285783 non-nul

In [18]:
# make and saving a new csv of the final data set (will not drop the original genre column)
df_merge_movies_final.to_csv('merged_movies_v1.csv', index=False)

## Conclusion

- This data is usable. 

- The timestamps added no significant value to the data. They identified when the data was submitted into the csv. There are also movie titles that are duplicated, they each have a different movieId and genre listings. They should be removed or decided if they should be combined under one movieId.

- The movie tags have a significantly lower amount of data, which only had 58 users that submitted tags across multiple movies. Therefore, many movies did not have tags and have empty values in those columns. At first glance, there doesn't seem to be any major issue because of the missing values.

Next steps:
- split the title column by the paranthesis and get a column for the year for additional analysis.
- Remove the duplicate movies titles or combine under one movieId.
- drop the original genres column.

## Streaming Movies and TV Dataset

The 4 Services Streaming Movies and TV dataset adds a platform and content-distribution perspective through metadata on films and television content available across Netflix, Hulu, Prime Video, and Disney+, including genre, release year, age rating, and IMDb scores linked to streaming platform strategy and audience accessibility.

In [19]:
# creating df for each csv
df_streaming = pd.read_csv("all_streaming.csv")
df_streaming_genre = pd.read_csv("all_genre.csv")
# df_streaming
# df_streaming_genre


In [20]:
# combining the data sets
df_movie_tv_streaming = df_streaming.join(df_streaming_genre)
df_movie_tv_streaming

,movie_or_serie,title,director,cast,country,date_added_platform,release_year,duration_seconds,gender_type,description,...,medical,night,policecop,sketch,soap,spyespionage,survival,opera,travel,melodrama
0,Movie,Duck the Halls: A Mickey Mouse Christmas Special,"Alonso Ramirez Ramos, Dave Wasson","Chris Diamantopoulos, Tony Anselmo, Tress MacN...",uninformed country,"November 26, 2021",2016,23 min,animation>family,Join Mickey and the gang as they duck the halls!,...,0,0,0,0,0,0,0,0,0,0
1,Movie,Ernest Saves Christmas,John Cherry,"Jim Varney, Noelle Parker, Douglas Seale",uninformed country,"November 26, 2021",1988,91 min,comedy,Santa Claus passes his magic bag to a new St. ...,...,0,0,0,0,0,0,0,0,0,0
2,Movie,Ice Age: A Mammoth Christmas,Karen Disher,"Raymond Albert Romano, John Leguizamo, Denis L...",United States,"November 26, 2021",2011,23 min,animation>comedy>family,Sid the Sloth is on Santa's naughty list.,...,0,0,0,0,0,0,0,0,0,0
3,Movie,The Queen Family Singalong,Hamish Hamilton,"Darren Criss, Adam Lambert, Derek Hough, Alexa...",uninformed country,"November 26, 2021",2021,41 min,musical,"This is real life, not just fantasy!",...,0,0,0,0,0,0,0,0,0,0
4,TV Show,The Beatles: Get Back,uninformed director,"John Lennon, Paul McCartney, George Harrison, ...",uninformed country,"November 25, 2021",2021,1 Season,docuseries>historical>music,A three-part documentary from Peter Jackson ca...,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22993,Movie,Pride Of The Bowery,Joseph H. Lewis,"Leo Gorcey, Bobby Jordan",uninformed country,NaN,1940,60 min,comedy,New York City street principles get an East Si...,...,0,0,0,0,0,0,0,0,0,0
22994,TV Show,Planet Patrol,uninformed director,"DICK VOSBURGH, RONNIE STEVENS, LIBBY MORRIS, M...",uninformed country,NaN,2018,4 Seasons,s,"This is Earth, 2100AD - and these are the adve...",...,0,0,0,0,0,0,0,0,0,0
22995,Movie,Outpost,Steve Barker,"Ray Stevenson, Julian Wadham, Richard Brake, M...",uninformed country,NaN,2008,90 min,action,"In war-torn Eastern Europe, a world-weary grou...",...,0,0,0,0,0,0,0,0,0,0
22996,TV Show,Maradona: Blessed Dream,uninformed director,"Esteban Recagno, Ezequiel Stremiz, Luciano Vit...",uninformed country,NaN,2021,1 Season,drama>sports,"The series tells the story of Diego Maradona, ...",...,0,0,0,0,0,0,0,0,0,0


In [21]:
# getting info and descriptive statistics
df_movie_tv_streaming.info()
df_movie_tv_streaming.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22998 entries, 0 to 22997
Columns: 122 entries, movie_or_serie to melodrama
dtypes: int64(112), object(10)
memory usage: 21.4+ MB


,release_year,Unnamed: 0,action,actionadventure,adult,adventure,animals,animation,anime,anthology,...,medical,night,policecop,sketch,soap,spyespionage,survival,opera,travel,melodrama
count,22998.000000,22998.000000,22998.000000,22998.000000,22998.000000,22998.000000,22998.000000,22998.000000,22998.000000,22998.000000,...,22998.000000,22998.000000,22998.000000,22998.000000,22998.000000,22998.000000,22998.000000,22998.000000,22998.000000,22998.000000
mean,2010.811244,0.002609,0.140838,0.019654,0.005392,0.080094,0.009044,0.048961,0.028698,0.001217,...,0.000261,0.000174,0.000043,0.000130,0.000087,0.000130,0.000391,0.000087,0.000043,0.000087
std,15.401142,0.051012,0.347862,0.138811,0.073232,0.271444,0.094672,0.215791,0.166960,0.034872,...,0.016150,0.013187,0.006594,0.011421,0.009325,0.011421,0.019779,0.009325,0.006594,0.009325
min,1920.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2010.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2016.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2019.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,2021.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [22]:
# checking if there are any movie duplicates under title
df_movie_tv_streaming['title'].is_unique


False

In [23]:
# show the duplicates
df_movie_tv_streaming[df_movie_tv_streaming['title'].duplicated(keep=False)]

,movie_or_serie,title,director,cast,country,date_added_platform,release_year,duration_seconds,gender_type,description,...,medical,night,policecop,sketch,soap,spyespionage,survival,opera,travel,melodrama
2,Movie,Ice Age: A Mammoth Christmas,Karen Disher,"Raymond Albert Romano, John Leguizamo, Denis L...",United States,"November 26, 2021",2011,23 min,animation>comedy>family,Sid the Sloth is on Santa's naughty list.,...,0,0,0,0,0,0,0,0,0,0
12,Movie,The Pixar Story,Leslie Iwerks,"Stacy Keach, John Lasseter, Brad Bird, John Mu...",United States,"November 19, 2021",2007,91 min,documentary>family,A groundbreaking company forever changes the f...,...,0,0,0,0,0,0,0,0,0,0
58,TV Show,PJ Masks,uninformed director,"Kyle Breitkopf, Jacob Ursomarzo, Addison Holley","France, United Kingdom","October 20, 2021",2015,5 Seasons,actionadventure>animation>kids,Look out Night Time Baddies the PJ Masks are c...,...,0,0,0,0,0,0,0,0,0,0
74,Movie,Black Widow,Cate Shortland,"Scarlett Johansson, Florence Pugh, David Harbo...",United States,"October 6, 2021",2021,135 min,actionadventure>science>fiction>spyespionage,Natasha confronts her history as a spy and the...,...,0,0,0,0,0,1,0,0,0,0
89,TV Show,Rolie Polie Olie,uninformed director,"Cole Caplan, Kristen Bone, Robert Smith, Cathe...","Canada, United States, France","September 29, 2021",1998,5 Seasons,animation>science>fiction,Rolie Polie Olie’s life and adventures center ...,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22950,Movie,The Rocky Horror Picture Show,Jim Sharman,"Richard O'Brien, Tim Curry, Patricia Quinn, Su...",uninformed country,NaN,1975,99 min,arts>entertainment>culture>comedy>science>fiction,"Part campy musical, part horror film, the movi...",...,0,0,0,0,0,0,0,0,0,0
22953,Movie,The Princess Bride,Rob Reiner,"Cary Elwes, Mandy Patinkin, Chris Sarandon, Ch...",uninformed country,NaN,1987,98 min,action>kids>science>fiction,Based on William Goldman's novel of the same n...,...,0,0,0,0,0,0,0,0,0,0
22979,Movie,Jack,Francis Ford Coppola,"Robin Williams, Diane Lane, Brian Kerwin, Jenn...",uninformed country,NaN,1996,113 min,comedy,Robin Williams stars as a 10-year-old in a gro...,...,0,0,0,0,0,0,0,0,0,0
22988,Movie,12 Dates of Christmas,uninformed director,uninformed cast,uninformed country,NaN,2011,87 min,drama>romance,"Setup on a Christmas Eve date, a woman must re...",...,0,0,0,0,0,0,0,0,0,0


After looking at the number of duplicates movies there are, the next step was to look and see if they were on different streaming platforms. that would give a "true" duplicate 

In [24]:
df_movie_tv_streaming[df_movie_tv_streaming.duplicated(subset=['title', 'channel_streaming'], keep=False)]

,movie_or_serie,title,director,cast,country,date_added_platform,release_year,duration_seconds,gender_type,description,...,medical,night,policecop,sketch,soap,spyespionage,survival,opera,travel,melodrama


In [25]:
# verification of there being no duplicates for movies on the same streaming service multiple times in the dataset
df_movie_tv_streaming.loc[df_movie_tv_streaming['title']== 'Black Widow', 'channel_streaming']

74             disney-movies-and-tv-shows
16379    amazon-prime-movies-and-tv-shows
Name: channel_streaming, dtype: object

In [26]:
# check for missing values
df_movie_tv_streaming[df_movie_tv_streaming.isnull().any(axis=1)]

,movie_or_serie,title,director,cast,country,date_added_platform,release_year,duration_seconds,gender_type,description,...,medical,night,policecop,sketch,soap,spyespionage,survival,opera,travel,melodrama
26,Movie,Marvel Studios’ 2021 Disney+ Day Special,uninformed director,uninformed cast,uninformed country,"November 12, 2021",2021,14 min,NaN,Marvel Studios’ Disney+ Day Special explores t...,...,0,0,0,0,0,0,0,0,0,0
30,Movie,Pixar 2021 Disney+ Day Special,uninformed director,"Pete Docter, Larry the Cable Guy, Jack Dylan G...",uninformed country,"November 12, 2021",2021,5 min,NaN,Join Pete Docter for a tour around Pixar and g...,...,0,0,0,0,0,0,0,0,0,0
339,Movie,Disney Holiday Magic Quest,uninformed director,uninformed cast,United States,"December 11, 2020",2020,46 min,NaN,ZOMBIES stars race to save the holiday magic!,...,0,0,0,0,0,0,0,0,0,0
1439,TV Show,Disney Kirby Buckets,uninformed director,"Jacob Bertrand, Mekai Curtis, Cade Sutton, Oli...",United States,NaN,2014,3 Seasons,actionadventure>comedy>coming>of>age,Welcome to Kirby's world! It's rude and sketchy.,...,0,0,0,0,0,0,0,0,0,0
1440,TV Show,Disney Mech-X4,uninformed director,"Nathaniel Potvin, Raymond Cham, Kamran Lucas, ...",Canada,NaN,2016,2 Seasons,actionadventure>comedy>science>fiction,Ryan discovers his ability to control a giant ...,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22993,Movie,Pride Of The Bowery,Joseph H. Lewis,"Leo Gorcey, Bobby Jordan",uninformed country,NaN,1940,60 min,comedy,New York City street principles get an East Si...,...,0,0,0,0,0,0,0,0,0,0
22994,TV Show,Planet Patrol,uninformed director,"DICK VOSBURGH, RONNIE STEVENS, LIBBY MORRIS, M...",uninformed country,NaN,2018,4 Seasons,s,"This is Earth, 2100AD - and these are the adve...",...,0,0,0,0,0,0,0,0,0,0
22995,Movie,Outpost,Steve Barker,"Ray Stevenson, Julian Wadham, Richard Brake, M...",uninformed country,NaN,2008,90 min,action,"In war-torn Eastern Europe, a world-weary grou...",...,0,0,0,0,0,0,0,0,0,0
22996,TV Show,Maradona: Blessed Dream,uninformed director,"Esteban Recagno, Ezequiel Stremiz, Luciano Vit...",uninformed country,NaN,2021,1 Season,drama>sports,"The series tells the story of Diego Maradona, ...",...,0,0,0,0,0,0,0,0,0,0


In [32]:
# want to show all columns
pd.set_option('display.max_rows', None)
print('Mean of Numerical Columns')
print(df_movie_tv_streaming.mean(numeric_only=True))

Mean of Numerical Columns
release_year       2010.811244
Unnamed: 0            0.002609
action                0.140838
actionadventure       0.019654
adult                 0.005392
adventure             0.080094
animals               0.009044
animation             0.048961
anime                 0.028698
anthology             0.001217
arthouse              0.006131
arts                  0.021002
biographical          0.001783
black                 0.004913
british               0.011001
buddy                 0.001739
cartoons              0.001478
children              0.027872
classic               0.006261
classics              0.001522
comedies              0.098052
comedy                0.160492
coming                0.008914
concert               0.000304
cooking               0.003479
crime                 0.030568
cult                  0.004305
dance                 0.000261
documentaries         0.060570
documentary           0.050744
docuseries            0.022480
drama        

In [33]:
print('Median of Numerical Columns')
print(df_movie_tv_streaming.median(numeric_only=True))

Median of Numerical Columns
release_year       2016.0
Unnamed: 0            0.0
action                0.0
actionadventure       0.0
adult                 0.0
adventure             0.0
animals               0.0
animation             0.0
anime                 0.0
anthology             0.0
arthouse              0.0
arts                  0.0
biographical          0.0
black                 0.0
british               0.0
buddy                 0.0
cartoons              0.0
children              0.0
classic               0.0
classics              0.0
comedies              0.0
comedy                0.0
coming                0.0
concert               0.0
cooking               0.0
crime                 0.0
cult                  0.0
dance                 0.0
documentaries         0.0
documentary           0.0
docuseries            0.0
drama                 0.0
dramas                0.0
faith                 0.0
family                0.0
fantasy               0.0
fitness               0.0
game      

In [34]:
print('Mode of Numerical Columns')
print(df_movie_tv_streaming.mode().iloc[0])

Mode of Numerical Columns
movie_or_serie                                    Movie
title                        10 Things I Hate About You
director                            uninformed director
cast                                    uninformed cast
country                              uninformed country
date_added_platform                   November 12, 2019
release_year                                     2019.0
duration_seconds                               1 Season
gender_type                                       drama
description                                           1
channel_streaming      amazon-prime-movies-and-tv-shows
Unnamed: 0                                          0.0
action                                              0.0
actionadventure                                     0.0
adult                                               0.0
adventure                                           0.0
animals                                             0.0
animation             

In [30]:
# making a duplicate df
df_movie_tv_stream = df_movie_tv_streaming.copy()
one_hot_2 = pd.get_dummies(df_movie_tv_stream['movie_or_serie'])

In [36]:
df_movie_tv_stream_final = df_movie_tv_stream.join(one_hot_2)
df_movie_tv_stream_final.sample(5)

,movie_or_serie,title,director,cast,country,date_added_platform,release_year,duration_seconds,gender_type,description,...,policecop,sketch,soap,spyespionage,survival,opera,travel,melodrama,Movie,TV Show
14268,TV Show,Sonic X,uninformed director,"Jason Griffith, Mike Pollock, Suzanne Goldish,...",uninformed country,NaN,2005,3 Seasons,animation>anime>kids,"Sonic, his friends, and the evil Dr. Eggman ar...",...,0,0,0,0,0,0,0,0,False,True
2534,TV Show,Heaven Official's Blessing,uninformed director,"Jiang Guangtao, Ma Zhengyang, Wen Sen, Hu Lian...",China,"April 9, 2021",2020,1 Season,anime>series>international>s>romantic>s,Banished to the mortal realm to exorcise ghost...,...,0,0,0,0,0,0,0,0,False,True
4531,Movie,How High 2,Bruce Leddy,"Lil Yachty, D.C. Young Fly, Alyssa Goss, DeRay...",United States,"December 31, 2019",2019,89 min,comedies,"When a pair of friends uncover a weed Bible, t...",...,0,0,0,0,0,0,0,0,True,False
10453,TV Show,Dancing With the Stars,uninformed director,uninformed cast,uninformed country,"September 21, 2021",2005,1 Season,family>lifestyle>culture>reality,ABC’s Dancing with the Stars is a spectacular ...,...,0,0,0,0,0,0,0,0,False,True
20033,Movie,The Terrible Adventure,Kel Thompson,"Olivia Thompson, Jackson Thompson, Ciro Dobric...",uninformed country,NaN,2021,94 min,comedy>kids,When two affluent siblings are faced with losi...,...,0,0,0,0,0,0,0,0,True,False


In [37]:
# creating a new csv
df_movie_tv_stream_final.to_csv("movie_tv_stream_v1.csv", index=False)

Conclusion

- This data is usable.

modify or correct the data in some way:
- Various rows are missing data but can be filled in with "uninformed" as many have already been filled in with.
- the duration_seconds column includes minutes and season number. This should be changed to "n/a" for tv seasons, and create a new column that is for tv season count. Also, this can be changed to minutes instead of seconds by removing the text in the row and changing it to a float/integer. 
- There doesn't appear to be any class imbalance.

Next Steps:
- missing data should be changed to be named "uninformed" to keep it similar across the dataset
- changing the duration_seconds column to duration_minutes and changing it to a float/int type
- creating a new column for number of tv seasons
- deeper analysis of release date versus added to the streaming service
- 0 and 1 streaming service column (?)

# 4. Storytelling With Data graph

Just like last week: choose any graph in the Introduction of Storytelling With Data (p. 1-17). Use matplotlib to reproduce it in a rough way. I don't expect you to spend an enormous amount of time on this; I understand that you likely will not have time to re-create every feature of the graph. However, if you're excited about learning to use matplotlib, this is a good way to do that. You don't have to duplicate the exact values on the graph; just the same rough shape will be enough.  If you don't feel comfortable using matplotlib yet, do the best you can and write down what you tried or what Google searches you did to find the answers.